In [0]:
from pyspark.sql.functions import col, max as spark_max

CATALOG = "dbw_ecommerce_om"
BRONZE_SCHEMA = "ecommerce"

ORDERS_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_orders"

print("Incremental processing setup loaded")
print(f"Source table: {ORDERS_TABLE}")

Incremental processing setup loaded
Source table: dbw_ecommerce_om.ecommerce.bronze_orders


In [0]:
bronze_orders = spark.table(ORDERS_TABLE)

latest_ingestion_timestamp = (
    bronze_orders
    .select(spark_max("_ingestion_timestamp").alias("latest_timestamp"))
    .collect()[0]["latest_timestamp"]
)

print(f"Latest Bronze ingestion timestamp: {latest_ingestion_timestamp}")

Latest Bronze ingestion timestamp: 2026-09-21 18:57:04.697650


In [0]:
from pyspark.sql.functions import current_timestamp

new_orders = spark.createDataFrame([
    (100101, 10001, "2026-09-22", "Completed", 1250.00),
    (100102, 10002, "2026-09-22", "Completed", 850.00),
    (100103, 10003, "2026-09-22", "Pending", 450.00),
    (100104, 10004, "2026-09-22", "Completed", 2100.00),
    (100105, 10005, "2026-09-22", "Pending", 675.00)
], [
    "order_id",
    "customer_id",
    "order_date",
    "status",
    "total_amount"
])

new_orders = new_orders.withColumn(
    "_ingestion_timestamp",
    current_timestamp()
)

display(new_orders)

order_id,customer_id,order_date,status,total_amount,_ingestion_timestamp
100101,10001,2026-09-22,Completed,1250.0,2026-09-22T16:31:55.218Z
100102,10002,2026-09-22,Completed,850.0,2026-09-22T16:31:55.218Z
100103,10003,2026-09-22,Pending,450.0,2026-09-22T16:31:55.218Z
100104,10004,2026-09-22,Completed,2100.0,2026-09-22T16:31:55.218Z
100105,10005,2026-09-22,Pending,675.0,2026-09-22T16:31:55.218Z


In [0]:
from pyspark.sql.functions import col

In [0]:
incremental_orders = new_orders.filter(
    col("_ingestion_timestamp") > latest_ingestion_timestamp
)

print(f"New records detected: {incremental_orders.count()}")

display(incremental_orders)

New records detected: 5


order_id,customer_id,order_date,status,total_amount,_ingestion_timestamp
100101,10001,2026-09-22,Completed,1250.0,2026-09-22T16:32:04.899Z
100102,10002,2026-09-22,Completed,850.0,2026-09-22T16:32:04.899Z
100103,10003,2026-09-22,Pending,450.0,2026-09-22T16:32:04.899Z
100104,10004,2026-09-22,Completed,2100.0,2026-09-22T16:32:04.899Z
100105,10005,2026-09-22,Pending,675.0,2026-09-22T16:32:04.899Z


In [0]:
incremental_orders.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dbw_ecommerce_om.ecommerce.bronze_orders")

print("Incremental batch appended successfully")

Incremental batch appended successfully


In [0]:
updated_count = spark.table(
    "dbw_ecommerce_om.ecommerce.bronze_orders"
).count()

print(f"Updated Bronze order count: {updated_count}")

Updated Bronze order count: 100105


In [0]:
silver_orders = spark.table(
    "dbw_ecommerce_om.silver.orders"
)

new_bronze_orders = spark.table(
    "dbw_ecommerce_om.ecommerce.bronze_orders"
).filter(
    col("_ingestion_timestamp") > latest_ingestion_timestamp
)

print(f"New Bronze orders to process: {new_bronze_orders.count()}")

display(new_bronze_orders)

New Bronze orders to process: 5


order_id,customer_id,order_date,status,total_amount,_ingestion_timestamp
100101,10001.0,2026-09-22,Completed,1250.0,2026-09-22T16:39:36.580Z
100102,10002.0,2026-09-22,Completed,850.0,2026-09-22T16:39:36.580Z
100103,10003.0,2026-09-22,Pending,450.0,2026-09-22T16:39:36.580Z
100104,10004.0,2026-09-22,Completed,2100.0,2026-09-22T16:39:36.580Z
100105,10005.0,2026-09-22,Pending,675.0,2026-09-22T16:39:36.580Z


In [0]:
from pyspark.sql.functions import col, trim, upper

new_silver_orders = (
    new_bronze_orders
    .withColumn("status", upper(trim(col("status"))))
    .filter(
        col("customer_id").isNotNull()
        & col("order_date").isNotNull()
        & (col("total_amount") > 0)
        & col("status").isin("COMPLETED", "PENDING", "CANCELLED")
    )
)

print(f"Valid Silver orders: {new_silver_orders.count()}")

display(new_silver_orders)

Valid Silver orders: 5


order_id,customer_id,order_date,status,total_amount,_ingestion_timestamp
100101,10001.0,2026-09-22,COMPLETED,1250.0,2026-09-22T16:39:36.580Z
100102,10002.0,2026-09-22,COMPLETED,850.0,2026-09-22T16:39:36.580Z
100103,10003.0,2026-09-22,PENDING,450.0,2026-09-22T16:39:36.580Z
100104,10004.0,2026-09-22,COMPLETED,2100.0,2026-09-22T16:39:36.580Z
100105,10005.0,2026-09-22,PENDING,675.0,2026-09-22T16:39:36.580Z


In [0]:
print("New incremental schema:")
new_silver_orders.printSchema()

print("\nExisting Silver schema:")
spark.table("dbw_ecommerce_om.silver.orders").printSchema()

New incremental schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)


Existing Silver schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: double (nullable = true)



In [0]:
from pyspark.sql.functions import col

new_silver_orders_aligned = (
    new_silver_orders
    .select(
        col("order_id").cast("int").alias("order_id"),
        col("customer_id").cast("int").alias("customer_id"),
        col("order_date"),
        col("status"),
        col("total_amount")
    )
)

print("Aligned incremental Silver schema:")
new_silver_orders_aligned.printSchema()

print(f"Records ready for Silver: {new_silver_orders_aligned.count()}")

display(new_silver_orders_aligned)

Aligned incremental Silver schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: double (nullable = true)

Records ready for Silver: 5


order_id,customer_id,order_date,status,total_amount
100101,10001,2026-09-22,COMPLETED,1250.0
100102,10002,2026-09-22,COMPLETED,850.0
100103,10003,2026-09-22,PENDING,450.0
100104,10004,2026-09-22,COMPLETED,2100.0
100105,10005,2026-09-22,PENDING,675.0


In [0]:
new_silver_orders_aligned.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dbw_ecommerce_om.silver.orders")

print("Incremental orders appended to Silver successfully")

Incremental orders appended to Silver successfully


In [0]:
updated_silver_count = spark.table(
    "dbw_ecommerce_om.silver.orders"
).count()

print(f"Updated Silver order count: {updated_silver_count}")

display(
    spark.table("dbw_ecommerce_om.silver.orders")
    .filter(col("order_id") >= 100101)
    .orderBy("order_id")
)


Updated Silver order count: 99605


order_id,customer_id,order_date,status,total_amount
100101,10001,2026-09-22,COMPLETED,1250.0
100101,7112,2026-06-01,COMPLETED,49848.53
100102,10002,2026-09-22,COMPLETED,850.0
100102,9379,2026-02-12,COMPLETED,5235.38
100103,10003,2026-09-22,PENDING,450.0
100103,8587,2026-03-19,PENDING,21194.43
100104,1108,2026-04-21,COMPLETED,3967.01
100104,10004,2026-09-22,COMPLETED,2100.0
100105,9590,2026-05-06,COMPLETED,35977.31
100105,10005,2026-09-22,PENDING,675.0


In [0]:
from pyspark.sql.functions import current_timestamp

new_order_items = spark.createDataFrame([
    (200001, 100101, 1, 2, 1250.00),
    (200002, 100102, 2, 1, 850.00),
    (200003, 100103, 3, 3, 150.00),
    (200004, 100104, 4, 2, 1050.00),
    (200005, 100105, 5, 1, 675.00)
], [
    "order_item_id",
    "order_id",
    "product_id",
    "quantity",
    "price"
])

new_order_items = new_order_items.withColumn(
    "_ingestion_timestamp",
    current_timestamp()
)

print(f"New incremental order items: {new_order_items.count()}")

display(new_order_items)

New incremental order items: 5


order_item_id,order_id,product_id,quantity,price,_ingestion_timestamp
200001,100101,1,2,1250.0,2026-09-22T16:59:30.194Z
200002,100102,2,1,850.0,2026-09-22T16:59:30.194Z
200003,100103,3,3,150.0,2026-09-22T16:59:30.194Z
200004,100104,4,2,1050.0,2026-09-22T16:59:30.194Z
200005,100105,5,1,675.0,2026-09-22T16:59:30.194Z


In [0]:
from pyspark.sql.functions import col

new_silver_order_items = (
    new_order_items
    .filter(
        col("product_id").isNotNull()
        & (col("quantity") > 0)
    )
)

print(f"Valid incremental order items: {new_silver_order_items.count()}")

display(new_silver_order_items)

Valid incremental order items: 5


order_item_id,order_id,product_id,quantity,price,_ingestion_timestamp
200001,100101,1,2,1250.0,2026-09-22T17:00:15.588Z
200002,100102,2,1,850.0,2026-09-22T17:00:15.588Z
200003,100103,3,3,150.0,2026-09-22T17:00:15.588Z
200004,100104,4,2,1050.0,2026-09-22T17:00:15.588Z
200005,100105,5,1,675.0,2026-09-22T17:00:15.588Z


In [0]:
new_order_items.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dbw_ecommerce_om.ecommerce.bronze_order_items")

print("Incremental order items appended to Bronze successfully")

Incremental order items appended to Bronze successfully


In [0]:
new_bronze_order_items = (
    spark.table("dbw_ecommerce_om.ecommerce.bronze_order_items")
    .filter(
        col("order_id").between(100101, 100105)
    )
)

print(f"New Bronze order items to process: {new_bronze_order_items.count()}")

display(
    new_bronze_order_items
    .orderBy("order_id")
)

New Bronze order items to process: 10


order_item_id,order_id,product_id,quantity,price,_ingestion_timestamp
583888,100101,1623.0,4,5918.39,2026-09-21T18:57:07.626Z
200001,100101,1.0,2,1250.0,2026-09-22T17:01:14.452Z
200002,100102,2.0,1,850.0,2026-09-22T17:01:14.452Z
504474,100103,346.0,2,20679.51,2026-09-21T18:57:07.626Z
200003,100103,3.0,3,150.0,2026-09-22T17:01:14.452Z
661971,100103,637.0,3,8240.68,2026-09-21T18:57:07.626Z
676553,100104,67.0,3,9703.75,2026-09-21T18:57:07.626Z
609028,100104,25.0,1,21683.83,2026-09-21T18:57:07.626Z
200004,100104,4.0,2,1050.0,2026-09-22T17:01:14.452Z
200005,100105,5.0,1,675.0,2026-09-22T17:01:14.452Z


In [0]:
order_items_bronze = spark.table(
    "dbw_ecommerce_om.ecommerce.bronze_order_items"
)

display(
    order_items_bronze
    .filter(col("order_id").between(100101, 100105))
    .select(
        "order_item_id",
        "order_id",
        "product_id",
        "quantity",
        "price",
        "_ingestion_timestamp"
    )
    .orderBy("_ingestion_timestamp", "order_item_id")
)

order_item_id,order_id,product_id,quantity,price,_ingestion_timestamp
504474,100103,346.0,2,20679.51,2026-09-21T18:57:07.626Z
583888,100101,1623.0,4,5918.39,2026-09-21T18:57:07.626Z
609028,100104,25.0,1,21683.83,2026-09-21T18:57:07.626Z
661971,100103,637.0,3,8240.68,2026-09-21T18:57:07.626Z
676553,100104,67.0,3,9703.75,2026-09-21T18:57:07.626Z
200001,100101,1.0,2,1250.0,2026-09-22T17:01:14.452Z
200002,100102,2.0,1,850.0,2026-09-22T17:01:14.452Z
200003,100103,3.0,3,150.0,2026-09-22T17:01:14.452Z
200004,100104,4.0,2,1050.0,2026-09-22T17:01:14.452Z
200005,100105,5.0,1,675.0,2026-09-22T17:01:14.452Z
